# The Thinking Atom

CognitiveWorker is the atomic building block of the Bridgic Amphibious framework — a pure thinking unit that only decides *what to do*, never *how to execute*. Combined with the `think_unit` descriptor and error strategies, it forms the foundation of every amphibious agent.

In this tutorial, we'll build a **travel planning assistant** that uses different CognitiveWorkers for different thinking responsibilities: one analyzes destinations, another creates itineraries.

## Initialize

In [ ]:
import os
from bridgic.model import OpenAILlm

api_key = os.environ.get("OPENAI_API_KEY", "your-api-key")
base_url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

llm = OpenAILlm(
    model="gpt-4o-mini",
    api_key=api_key,
    base_url=base_url,
)

Let's define some travel-related tools.

In [ ]:
from bridgic.core import tool

@tool(description="Search for tourist attractions in a city")
async def search_attractions(city: str) -> str:
    attractions = {
        "Tokyo": "Senso-ji Temple, Shibuya Crossing, Tokyo Tower, Meiji Shrine",
        "Kyoto": "Fushimi Inari Shrine, Kinkaku-ji, Arashiyama Bamboo Grove",
        "Osaka": "Osaka Castle, Dotonbori, Universal Studios Japan",
    }
    return attractions.get(city, f"No attraction data for {city}")

@tool(description="Get estimated travel cost between two cities")
async def get_travel_cost(origin: str, destination: str) -> str:
    return f"Estimated cost from {origin} to {destination}: $150-300 (train/flight)"

@tool(description="Check hotel availability in a city for given dates")
async def check_hotels(city: str, checkin: str, checkout: str) -> str:
    return f"3 hotels available in {city} from {checkin} to {checkout}, starting at $80/night"

## Part 1: CognitiveWorker — The Pure Thinking Unit

In the observe-think-act cycle, a `CognitiveWorker` handles the **think** phase. It receives context (what the agent knows) and decides what action to take next — but never executes the action itself.

### Two Ways to Create a CognitiveWorker

**Method 1: `CognitiveWorker.inline()`** — for simple cases where a prompt string is enough.

In [ ]:
from bridgic.amphibious import CognitiveWorker

# Quick creation with a prompt string
simple_worker = CognitiveWorker.inline(
    "Analyze the travel destination and suggest the best attractions to visit."
)

**Method 2: Subclass `CognitiveWorker`** — for custom observation and thinking logic.

In [ ]:
from bridgic.amphibious import CognitiveContext

class DestinationAnalyzer(CognitiveWorker):
    """A worker that analyzes destinations with custom observation."""

    async def thinking(self) -> str:
        return "Analyze the destination based on the observation and suggest a day-by-day plan."

    async def observation(self, context: CognitiveContext):
        """Inject extra environmental information before thinking."""
        return (
            f"Current goal: {context.goal}\n"
            f"Travel tip: Consider visiting attractions early morning to avoid crowds."
        )

The `observation()` method lets you inject additional context before the worker thinks. By default, it delegates to the agent-level observation. When you override it, the worker gets a custom perception of its environment.

## Part 2: think_unit — Declarative Execution Configuration

A `think_unit` binds a CognitiveWorker with execution parameters and declares it as a class attribute on the agent. Each `await self.think_unit` triggers one full observe-think-act cycle.

In [ ]:
from bridgic.amphibious import AmphibiousAutoma, CognitiveContext, think_unit

class TravelPlanner(AmphibiousAutoma[CognitiveContext]):
    # Declare think units as class attributes
    analyzer = think_unit(
        CognitiveWorker.inline("Analyze the destination and find key attractions."),
        max_attempts=3,
    )

    planner = think_unit(
        CognitiveWorker.inline("Create a detailed day-by-day itinerary based on what we know."),
        max_attempts=5,
    )

    async def on_agent(self, ctx: CognitiveContext):
        # Each await triggers one observe-think-act cycle
        await self.analyzer
        await self.planner

In [ ]:
agent = TravelPlanner(llm=llm)
result = await agent.arun(
    goal="Plan a 3-day trip to Tokyo",
    tools=[search_attractions, get_travel_cost, check_hotels],
)
print(result)

### Conditional Loops with `until()`

Use `until()` to run a think_unit repeatedly until a condition is met. This is useful when the agent needs multiple cycles to complete a task.

In [ ]:
class IterativePlanner(AmphibiousAutoma[CognitiveContext]):
    researcher = think_unit(
        CognitiveWorker.inline(
            "Research ONE aspect of the trip (attractions, costs, or hotels). "
            "Focus on what hasn't been researched yet."
        ),
        max_attempts=10,
    )

    async def on_agent(self, ctx: CognitiveContext):
        # Run until the agent decides the research is complete
        await self.researcher

agent = IterativePlanner(llm=llm)
result = await agent.arun(
    goal="Research a 3-day trip to Kyoto: find attractions, estimate costs, and check hotels for June 15-18",
    tools=[search_attractions, get_travel_cost, check_hotels],
)
print(result)

### Filtering Available Tools

You can restrict which tools are visible to a think_unit using the `tools` parameter. This focuses the worker's attention.

In [ ]:
class FocusedPlanner(AmphibiousAutoma[CognitiveContext]):
    # This worker can only see search_attractions
    attraction_finder = think_unit(
        CognitiveWorker.inline("Find the best attractions to visit."),
        max_attempts=3,
    )

    # This worker can see all tools
    full_planner = think_unit(
        CognitiveWorker.inline("Create a complete travel plan with costs and hotels."),
        max_attempts=5,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.attraction_finder.until(
            lambda ctx: True,  # Run once
            tools=["search_attractions"],
        )
        await self.full_planner

agent = FocusedPlanner(llm=llm)
result = await agent.arun(
    goal="Plan a trip to Osaka",
    tools=[search_attractions, get_travel_cost, check_hotels],
)
print(result)

## Part 3: ErrorStrategy — When Things Go Wrong

When a tool call fails or an error occurs during the think cycle, `ErrorStrategy` determines what happens next.

In [ ]:
from bridgic.amphibious import ErrorStrategy

call_count = 0

@tool(description="Book a hotel room (may fail due to network issues)")
async def book_hotel(city: str, hotel_name: str) -> str:
    global call_count
    call_count += 1
    if call_count <= 2:
        raise ConnectionError(f"Network timeout when booking {hotel_name}")
    return f"Successfully booked {hotel_name} in {city}!"

In [ ]:
class ResilientBooker(AmphibiousAutoma[CognitiveContext]):
    booker = think_unit(
        CognitiveWorker.inline("Book the hotel that best fits the budget."),
        max_attempts=5,
        on_error=ErrorStrategy.RETRY,
        max_retries=3,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.booker

call_count = 0  # Reset counter
agent = ResilientBooker(llm=llm)
result = await agent.arun(
    goal="Book a hotel in Tokyo",
    tools=[book_hotel, check_hotels],
)
print(result)

The three error strategies:

- **`ErrorStrategy.RAISE`** (default): Re-raises the exception immediately. Use when failures should stop execution.
- **`ErrorStrategy.IGNORE`**: Silently skips the failed step and continues. Use when individual failures are acceptable.
- **`ErrorStrategy.RETRY`**: Automatically retries up to `max_retries` times. Use for transient errors like network timeouts.

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we explored the fundamental building blocks of the Bridgic Amphibious framework:

- **CognitiveWorker** is a pure thinking unit in the observe-think-act cycle. It only decides *what to do* — execution is handled by the framework. Create one quickly with `CognitiveWorker.inline("prompt")` or subclass it for custom `thinking()` and `observation()` logic.
- **think_unit** is a declarative descriptor that binds a worker with execution parameters (`max_attempts`, `on_error`, `max_retries`). Declare it as a class attribute on your agent.
- **`await self.think_unit`** triggers one complete observe-think-act cycle. Use **`until()`** to loop until a condition is met, and **`tools=`/`skills=`** to filter what the worker can see.
- **ErrorStrategy** controls failure behavior: `RAISE` stops on error, `IGNORE` skips errors, `RETRY` automatically retries transient failures.

Next, we'll learn how to orchestrate think units using the two modes: `on_agent` for LLM-driven logic and `on_workflow` for deterministic steps.